# Shinhan Card 파일 불러오기

## 0. 초기 세팅
각종 라이브러리를 불러오고 초기값 등을 세팅합니다.

In [1]:
#!uv add xlrd
#!uv add lxml

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
from ledgerly.expenditure import (shinhan_file_config
                                  , map_shinhan_card_df_to_expenditure
                                  , preprocess_shinhan_data
                                  , map_category
                                  , insert_expenditure_data
                                  , fetch_expenditure_data)


## 1. 데이터 파일 조회 및 전처리

In [4]:
# 데이터 파일 경로 설정 (가변적)
target_yymm = "2603" # 예: "2603", 없을 경우 빈 문자열 ""
target_filename = "shinhancard_2602.xls"

if target_yymm:
    target_file = shinhan_file_config["data_dir"] / target_yymm / target_filename
else:
    target_file = shinhan_file_config["data_dir"] / target_filename

sh_ori_df = pd.read_excel(target_file)
len(sh_ori_df)


5

In [5]:
sh_df = preprocess_shinhan_data(sh_ori_df)
sh_df.head()

,거래일,카드구분,이용카드,가맹점명,승인번호,금액,매입구분,이용구분,거래통화,해외이용금액,취소상태
0,2026-02-28 15:05:00,신용,본인204*,홈플러스㈜,34650996,88750,결제확정,일시불,NaN,NaN,NaN
1,2026-02-21 17:18:00,신용,본인204*,(주)지에스리테일 GS수퍼 김포한,41118038,5820,결제확정,일시불,NaN,NaN,NaN
2,2026-02-21 17:13:00,신용,본인204*,미소유통,41046345,10000,결제확정,일시불,NaN,NaN,NaN
3,2026-02-07 12:01:00,신용,본인204*,홈플러스㈜,5400560,117320,결제확정,일시불,NaN,NaN,NaN


## 2. DB 데이터로 매핑

In [6]:
expenditure_df = map_shinhan_card_df_to_expenditure(sh_df)
expenditure_df.head()

,used_at,payment_type,payment_provider,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid
0,2026-02-28 15:05:00,card,shinhan,홈플러스㈜,single,88750,0,unknown,None,card_shinhan_34650996
1,2026-02-21 17:18:00,card,shinhan,(주)지에스리테일 GS수퍼 김포한,single,5820,0,unknown,None,card_shinhan_41118038
2,2026-02-21 17:13:00,card,shinhan,미소유통,single,10000,0,unknown,None,card_shinhan_41046345
3,2026-02-07 12:01:00,card,shinhan,홈플러스㈜,single,117320,0,unknown,None,card_shinhan_5400560


In [7]:
expenditure_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   used_at           4 non-null      datetime64[ns]
 1   payment_type      4 non-null      object        
 2   payment_provider  4 non-null      object        
 3   merchant_name     4 non-null      string        
 4   installment_type  4 non-null      object        
 5   amount            4 non-null      int64         
 6   remaining_amount  4 non-null      int64         
 7   category          4 non-null      object        
 8   memo              0 non-null      object        
 9   source_uid        4 non-null      object        
dtypes: datetime64[ns](1), int64(2), object(6), string(1)
memory usage: 452.0+ bytes


## 3. 지출 분류

In [8]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df["category"] = expenditure_df["category"].fillna("미분류")

In [9]:
expenditure_df[["merchant_name", "category"]].head(20)

,merchant_name,category
0,홈플러스㈜,식비
1,(주)지에스리테일 GS수퍼 김포한,식비
2,미소유통,기타
3,홈플러스㈜,식비


## 4. DB 저장

In [10]:
insert_expenditure_data(expenditure_df)
print(f"{len(expenditure_df)}개의 데이터가 성공적으로 저장되었습니다.")

4개의 데이터가 성공적으로 저장되었습니다.


In [11]:
fetched_db_df = fetch_expenditure_data()

In [12]:
fetched_db_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187 entries, 0 to 186
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                187 non-null    int64 
 1   used_at           187 non-null    object
 2   payment_type      187 non-null    object
 3   payment_provider  187 non-null    object
 4   merchant_name     187 non-null    object
 5   installment_type  187 non-null    object
 6   amount            187 non-null    object
 7   remaining_amount  187 non-null    int64 
 8   category          187 non-null    object
 9   memo              15 non-null     object
 10  source_uid        187 non-null    object
 11  created_at        187 non-null    object
dtypes: int64(2), object(10)
memory usage: 17.7+ KB
